# NER Matching — Exploration

Explore `data/results.parquet` produced by `run_match.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_parquet('data/results.parquet')
delegates = pd.read_parquet('data/delegates_reference.parquet')

print(f'Results: {len(results):,} rows')
print(f'Columns: {results.columns.tolist()}')
results.head(5)

## Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results['score_1'].hist(bins=50, ax=ax, edgecolor='none')
ax.axvline(0.2, color='red', linestyle='--', label='threshold=0.2')
ax.set_xlabel('Top-1 score')
ax.set_ylabel('Count')
ax.set_title('Distribution of best-match scores')
ax.legend()
plt.tight_layout()
plt.show()

print(results['score_1'].describe())

## Match rate by year

In [ ]:
threshold = 0.2
results['matched'] = results['score_1'] >= threshold

by_year = results.groupby('year').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
).reset_index()
by_year['match_rate'] = by_year['matched'] / by_year['total']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(by_year['year'], by_year['match_rate'], marker='.', linewidth=1)
ax.set_xlabel('Year')
ax.set_ylabel(f'Match rate (score ≥ {threshold})')
ax.set_title('Match rate by year')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

overall = results['matched'].mean()
print(f'Overall match rate: {overall:.1%}  ({results["matched"].sum():,} / {len(results):,})')

## Top unmatched spans

Spans with score_1 < 0.1 — candidates for new delegates or noise.

In [ ]:
unmatched = results[results['score_1'] < 0.1].copy()
print(f'Unmatched spans: {len(unmatched):,} ({len(unmatched)/len(results):.1%})')

top_unmatched = (
    unmatched.groupby('tag_text')
    .agg(count=('year', 'count'), years=('year', lambda x: sorted(x.unique())))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)
top_unmatched

## Sample: span + top-3 candidates

In [ ]:
# Join cand_1 to fullname
id_to_name = delegates.set_index('cons_id_str')['fullname'].to_dict()

sample = results[results['score_1'] >= threshold].sample(min(20, len(results)), random_state=42).copy()
for k in range(1, 4):
    col = f'cand_{k}'
    if col in sample.columns:
        sample[f'name_{k}'] = sample[col].map(lambda x: id_to_name.get(str(x), '') if pd.notna(x) else '')

display_cols = ['tag_text', 'year', 'score_1', 'name_1', 'score_2', 'name_2', 'score_3', 'name_3']
sample[[c for c in display_cols if c in sample.columns]].sort_values('score_1', ascending=False)